# Merge and Export Machine Learning Dataset
This notebook gathers all the pre-calculated features from the other scripts (OSM POIs, Land Use percentages, and Terrain Slope) and merges them into one single comprehensive `.csv` table that will be fed to the Machine Learning models.

## Note: Data Requirements

In [8]:
import pandas as pd
import os

print("Libraries imported.")

Libraries imported.


## Load the datatsets

In [9]:
# Define file paths (UNCOMMENT AND ADJUST ONCE YOU HAVE EXPORTED THE DATA)

path_osm_roads = "data/osm_roads_features.csv"
path_momepy_roads = "data/momepy_roads_features.csv"
path_osm_points = "data/osm_points_features.csv"

df_osm_roads = pd.read_csv(path_osm_roads)
df_momepy_roads = pd.read_csv(path_momepy_roads)
df_osm_points = pd.read_csv(path_osm_points)

print("Data loaded successfully.")

Data loaded successfully.


In [10]:
df_osm_roads.head()



,road_id,noise_day,noise_evening,noise_night,road_length,road_category,dist_to_main
0,13845167536_847716886_0,55,55,50,401.483629,5,258.681126
1,21638869_30343583_0,55,50,50,90.289544,5,429.190460
2,21638869_242318729_0,55,55,50,15.055894,4,512.333789
3,21638869_30343672_0,60,60,55,15.404960,4,516.423077
4,3055187877_431100338_0,60,60,55,46.142160,5,214.041852


In [11]:
df_momepy_roads.head()

,u,v,key,osmid,highway,lanes,maxspeed,name,oneway,reversed,...,tunnel,bridge,service,segment_id,geometry,openness,width_deviation,height,height_deviation,hw_ratio
0,21638845,885308258,0,554707785,residential,2,30,Passeig de Joan de Borbó,True,False,...,NaN,NaN,NaN,21638845_885308258_0,LINESTRING (432071.1859596025 4580160.40762989...,0.750000,5.740353,4.500000,2.121320,0.104589
1,21638845,13175411896,0,"[1433594636, 4079500, 543357596, 902517725, 42...",residential,"['3', '4']",30,Passeig de Joan de Borbó,False,True,...,NaN,NaN,NaN,21638845_13175411896_0,LINESTRING (432071.1859596025 4580160.40762989...,0.363095,5.540738,12.700935,7.896996,0.486348
2,21638865,30343650,0,4079505,living_street,NaN,20,Carrer del Judici,True,False,...,NaN,NaN,NaN,21638865_30343650_0,LINESTRING (432319.661133097 4580899.827105286...,0.750000,4.211717,11.500000,12.020815,0.759783
3,21638867,1636238503,0,4079509,tertiary,2,30,Carrer de Pepe Rubianes,False,True,...,NaN,NaN,NaN,21638867_1636238503_0,LINESTRING (432179.1460232024 4581060.54342001...,0.666667,0.210299,19.000000,1.414214,1.046582
4,21638867,1403082237,0,4079509,tertiary,2,30,Carrer de Pepe Rubianes,False,False,...,NaN,NaN,NaN,21638867_1403082237_0,LINESTRING (432179.1460232024 4581060.54342001...,0.666667,1.308920,19.000000,1.414214,0.942525


## Merge dataframes

In [12]:
# Function to merge feature DataFrame with base DataFrame

def merge_feature_dataframe(base_df, feature_df, cols_to_drop):
    # Drop target columns
    clean_df = feature_df.drop(columns=[c for c in cols_to_drop if c in feature_df.columns], errors='ignore')
    
    # Drop duplicate columns (except road_id)
    cols_to_remove = [c for c in clean_df.columns if c in base_df.columns and c != 'road_id']
    clean_df = clean_df.drop(columns=cols_to_remove, errors='ignore')
    
    # Ensure road_id exists and is consistent
    if 'road_id' not in clean_df.columns or 'road_id' not in base_df.columns:
        return base_df
    
    clean_df['road_id'] = clean_df['road_id'].astype(str)
    base_df = base_df.copy()
    base_df['road_id'] = base_df['road_id'].astype(str)
    
    return base_df.merge(clean_df, on='road_id', how='left')

In [13]:

ml_dataset = df_osm_roads.copy()

# Columns to remove to avoid repetition
cols_to_drop = ['noise_day', 'noise_evening', 'noise_night']

# Check road_id types before merging
print("Road ID types:")
print(f"ml_dataset: {ml_dataset['road_id'].dtype}")

for name, df in [
    ('OSM Points', df_osm_points),
    ('Momepy Roads', df_momepy_roads),
]:
    if 'road_id' in df.columns:
        print(f"{name}: {df['road_id'].dtype}")
    else:
        print(f"{name}: NO road_id column!")
    ml_dataset = merge_feature_dataframe(ml_dataset, df, cols_to_drop)

ml_dataset = ml_dataset.fillna(0)

display(ml_dataset.head(10))
ml_dataset.info()
print(f"Final dataset shape: {ml_dataset.shape}")

Road ID types:
ml_dataset: str
OSM Points: str
Momepy Roads: NO road_id column!


,road_id,noise_day,noise_evening,noise_night,road_length,road_category,dist_to_main,signals_50,transport_100,pois_noisy_100,pois_sensitive_100
0,13845167536_847716886_0,55,55,50,401.483629,5,258.681126,1,10,3,1
1,21638869_30343583_0,55,50,50,90.289544,5,429.190460,1,3,30,1
2,21638869_242318729_0,55,55,50,15.055894,4,512.333789,1,3,21,1
3,21638869_30343672_0,60,60,55,15.404960,4,516.423077,1,3,22,1
4,3055187877_431100338_0,60,60,55,46.142160,5,214.041852,1,2,1,2
5,3109574424_3109574426_0,55,50,45,75.757316,6,38.702252,1,3,16,1
6,21638867_30343414_0,65,60,55,91.037745,5,382.085637,1,5,23,2
7,21638878_242317164_0,40,40,0,14.636801,5,420.707644,1,3,22,1
8,21638877_30343573_0,50,50,45,59.188192,5,341.179073,1,2,29,2
9,21638877_1636238506_0,50,50,45,14.928692,5,396.819559,1,2,22,2


<class 'pandas.DataFrame'>
RangeIndex: 11670 entries, 0 to 11669
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   road_id             11670 non-null  str    
 1   noise_day           11670 non-null  int64  
 2   noise_evening       11670 non-null  int64  
 3   noise_night         11670 non-null  int64  
 4   road_length         11670 non-null  float64
 5   road_category       11670 non-null  int64  
 6   dist_to_main        11670 non-null  float64
 7   signals_50          11670 non-null  int64  
 8   transport_100       11670 non-null  int64  
 9   pois_noisy_100      11670 non-null  int64  
 10  pois_sensitive_100  11670 non-null  int64  
dtypes: float64(2), int64(8), str(1)
memory usage: 1003.0 KB
Final dataset shape: (11670, 11)


### Export to CSV
Finally, we dump out the master table into a standard format readable by `scikit-learn` in the next phase!

In [15]:
# UNCOMMENT TO EXPORT

output_dir = "data"
os.makedirs(output_dir, exist_ok=True)

final_path = os.path.join(output_dir, "bcn_noise_ml_dataset.csv")
ml_dataset.to_csv(final_path, index=False)

print(f"Dataset successfully exported to: {final_path}")

Dataset successfully exported to: data\bcn_noise_ml_dataset.csv
